# Step 14 — Qwen2.5-7B LoRA fine-tune (out-of-family anchor, generative regression)

**Why this exists:** after step 13 (E5 4-anchor → Public LB **0.72394**), L15 documented that the contrastive-sentence-similarity family (SPECTER2 / SciNCL / BGE / E5, all r > 0.91 to each other) has reached its ceiling. Each additional encoder in this family lifts +0.0001-0.0003 public per +0.005-0.015 OOF (13× discount). To break the ceiling we need a model with **fundamentally different signal geometry** — generative regression instead of CLS/mean pooling.

**Why Qwen2.5-7B specifically:**
- 7B parameters with Apache 2.0 license — fits A100 40GB at 4-bit + LoRA, fits Kaggle T4 16GB at 4-bit (slightly slower).
- Strong on niche-domain instruction following (top of MMLU + zero-shot reasoning benchmarks).
- Different lineage: causal-LM, not bidirectional encoder. Pearson r vs E5 expected in 0.6-0.8 band — the diversity sweet spot per L9.
- LLM zero-shot (L8) failed at OOF 0.37 because the model "played safe" toward labels 1-2 on a niche topic. **Fine-tuning** lets the model recalibrate to the actual ASP-relevance scale from the 2,494 labelled training rows.

**Recipe — SFT with verbalizer inference (the LoRA SFT community standard):**
- Model: `Qwen/Qwen2.5-7B-Instruct` loaded in 4-bit NF4 (bitsandbytes)
- LoRA: r=16, alpha=32, dropout=0.05, target modules `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`
- Prompt: chat template with system message defining the 1-5 ASP-relevance scale + user message with title+abstract + assistant message containing only the digit
- Training: standard next-token cross-entropy with prompt tokens masked to -100 (loss on answer digit only)
- Optimizer: paged AdamW 8-bit, LR `2e-4` (LoRA standard), weight decay `0.01`, **3 epochs** (avoid overfit on 2,494 samples)
- Schedule: cosine + 5% warmup, grad clip 1.0, bf16 (fp16 fallback)
- Batch: `BATCH_TRAIN=4`, `GRAD_ACCUM_STEPS=4` (effective batch = 16, matches encoder anchors)
- 5 folds × 1 seed = 5 models (default; extendable to 3 seeds in cell 15)
- **Verbalizer inference:** at the assistant position, take logits over single tokens `1`, `2`, `3`, `4`, `5`, softmax, return `sum(p_i * i)` as the continuous score. This makes the output regression-compatible for stacking.

**Decision after fold 1 (per L11/L13):**
- best per-fold round-QWK ≥ 0.55 → continue, anchor candidate worth full 5-fold run
- best per-fold 0.50-0.55 → continue but expect borderline blend probe
- best per-fold < 0.50 → abort, drop step 14, signal floor failed

**Decision after full 5-fold (per L11/L13/L15):**
- Compute Pearson r vs E5 OOF.
  - r > 0.93 → same-family redundancy (unlikely for generative model, but possible if the LM converges to the same label calibration). Drop.
  - r 0.7-0.92 → diversity sweet spot. Run +10% blend probe vs `safest_e5` (50/30/20/00).
  - r < 0.7 → diversity excellent. Probe more aggressive weights (up to 0.35).
- If +10% blend probe lifts OOF round-QWK by ≥ 0.005 → build 5-anchor (or 4-anchor with Qwen replacing one) sweep.
- Otherwise → drop, codify L16.

**Time budget on A100 40GB:**
- Model load (4-bit + LoRA prep): ~1 min
- Train per fold: 3 epochs × ~5 min = ~15 min
- Predict per fold (val + public + private): ~2 min
- Total: ~17 min/fold × 5 folds = **~85-95 min for 1 seed (5 models)**.
- For 3 seeds (15 models): ~4.5h — set in cell 15.

**Kaggle T4 fallback:** 4-bit Qwen-7B fits in 8GB. Set `BATCH_TRAIN=2, GRAD_ACCUM_STEPS=8` (effective 16 unchanged). Per-step ~3× slower → ~4-5h for 1 seed.


## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.45" "accelerate>=0.34" "peft>=0.13" "bitsandbytes>=0.44" "sentencepiece>=0.2" scikit-learn pandas "numpy<2" scipy

## 2. Data setup (Colab / Kaggle / local)

Auto-detects platform and locates `asp_data*.zip`. Same logic as step 9/12.

Source priority:
1. `ASP_DATA_ZIP` env var
2. `asp_data*.zip` in cwd or any parent (up to repo root)
3. Kaggle: any `*.zip` under `/kaggle/input/`
4. Colab: interactive upload widget (fallback)

In [ ]:
import os, pathlib, zipfile, shutil

def _detect_platform():
    if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
        return 'colab'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or pathlib.Path('/kaggle/working').exists():
        return 'kaggle'
    return 'local'

PLATFORM = _detect_platform()
print('platform =', PLATFORM)

if PLATFORM == 'colab':
    WORK = pathlib.Path('/content/work')
elif PLATFORM == 'kaggle':
    WORK = pathlib.Path('/kaggle/working/asp_work')
else:
    WORK = pathlib.Path.cwd() / 'work'

DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'qwen_lora_finetune'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _find_local_zip():
    env_zip = os.environ.get('ASP_DATA_ZIP')
    if env_zip and pathlib.Path(env_zip).exists():
        return pathlib.Path(env_zip)
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        for pat in ('asp_data*.zip', 'asp_data.zip'):
            for cand in sorted(base.glob(pat)):
                return cand
        if (base / '.git').exists():
            break
    return None

def _find_kaggle_zip():
    root = pathlib.Path('/kaggle/input')
    if not root.exists():
        return None
    for cand in sorted(root.rglob('asp_data*.zip')):
        return cand
    for cand in sorted(root.rglob('*.zip')):
        return cand
    return None

def _ingest_zip(zip_path):
    print('using zip:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA)
    print('extracted ->', DATA)

zip_path = _find_local_zip()
if zip_path is None and PLATFORM == 'kaggle':
    zip_path = _find_kaggle_zip()

if zip_path is not None:
    _ingest_zip(zip_path)
elif PLATFORM == 'colab':
    from google.colab import files
    print('No local zip found; falling back to Colab upload widget.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = WORK / name
        target.write_bytes(content)
        if name.lower().endswith('.zip'):
            _ingest_zip(target)
else:
    raise FileNotFoundError(
        'No data zip found. Set ASP_DATA_ZIP=/path/to/asp_data.zip, place '
        'asp_data*.zip in cwd or a parent, or attach the dataset on Kaggle.')

print('\nFiles in DATA:')
for p in sorted(DATA.glob('*')):
    print(' ', p.name, p.stat().st_size)

## 3. Load + merge

In [ ]:
import pandas as pd, numpy as np, json, re, time
from pathlib import Path

DATA = Path(DATA)
RUN_DIR = Path(RUN_DIR)

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('using abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 4. Prompt template + tokenizer

The prompt structure follows the L5 / EDA findings about what the labels actually mean:
- Label 1 = proceedings volume / off-topic / summary
- Label 2 = adjacent KR/verification, no ASP
- Label 3 = generic logic / declarative reasoning
- Label 4 = applied or extending ASP
- Label 5 = core ASP advance or ASP×AI cross-over

The system prompt encodes those scale anchors so the LM has a clear rubric. Training only computes loss on the single answer-digit token; the prompt tokens are masked to -100.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
MAX_LEN = 768          # ~600 token prompt + small headroom; abstracts up to ~500 tokens
BATCH_TRAIN = 4        # 4-bit Qwen-7B + LoRA peaks ~22GB on A100 40GB at this
BATCH_EVAL = 8
GRAD_ACCUM_STEPS = 4   # effective batch = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # left-pad for causal LM (so generation/inference position is consistent)

# Verify the digit tokens '1'..'5' are single tokens — verbalizer requires this.
LABEL_TOKEN_IDS = []
for i in range(1, 6):
    ids = tokenizer.encode(str(i), add_special_tokens=False)
    assert len(ids) == 1, f"label '{i}' must tokenize to 1 token, got {ids}"
    LABEL_TOKEN_IDS.append(ids[0])
print('label token ids =', dict(zip(range(1, 6), LABEL_TOKEN_IDS)))

SYSTEM_PROMPT = (
    "You are a research analyst evaluating papers in Answer Set Programming (ASP) "
    "and AI symbolic reasoning. Score each paper on a 1-5 scale based on how directly "
    "it advances ASP/AI-symbolic methods:\n"
    "1 = proceedings volume, off-topic summary, or workshop overview\n"
    "2 = adjacent topic (KR, verification, planning) without ASP\n"
    "3 = generic logic / declarative reasoning, not ASP-specific\n"
    "4 = applied or extending ASP (clingo/dlv users, ASP solvers)\n"
    "5 = core ASP advance or ASP \u00d7 AI cross-over (neuro-symbolic, NN verification, "
    "explainable AI built on ASP, ASP semantics theory)\n"
    "Respond with the single digit 1, 2, 3, 4, or 5. Nothing else."
)

def build_messages(title, abstract, label=None):
    title = '' if pd.isna(title) else str(title).strip()
    abstract = '' if pd.isna(abstract) else str(abstract).strip()
    user = f"Title: {title}\n\nAbstract: {abstract}" if abstract else f"Title: {title}"
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user},
    ]
    if label is not None:
        msgs.append({'role': 'assistant', 'content': str(int(label))})
    return msgs

def render_prompt(title, abstract, with_label=False, label=None):
    msgs = build_messages(title, abstract, label if with_label else None)
    if with_label:
        # Full conversation incl. assistant answer (used for SFT training)
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    # Prompt only — model will be conditioned on this and we read logits at next position (inference)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

class PaperDataset(Dataset):
    """Training mode: returns input_ids + labels with prompt masked to -100.
    Inference mode (with_label=False): returns input_ids + position to read logits at.
    """
    def __init__(self, df, with_label):
        self.with_label = with_label
        self.titles = df['title'].tolist()
        self.abstracts = df['abstract'].tolist()
        self.labels = df['Label'].astype(int).tolist() if with_label else [0] * len(df)

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        return {
            'title': self.titles[idx],
            'abstract': self.abstracts[idx],
            'label': self.labels[idx],
            'idx': idx,
        }

def collate_train(batch):
    """Tokenize full conversation; mask everything before the answer digit to -100."""
    full_texts = [render_prompt(b['title'], b['abstract'], with_label=True, label=b['label']) for b in batch]
    prompt_only = [render_prompt(b['title'], b['abstract'], with_label=False) for b in batch]

    enc_full = tokenizer(full_texts, padding=True, truncation=True, max_length=MAX_LEN,
                         return_tensors='pt', add_special_tokens=False)
    # Find prompt length per row to mask labels
    enc_prompt = tokenizer(prompt_only, padding=False, truncation=True, max_length=MAX_LEN,
                           add_special_tokens=False)
    prompt_lens = [len(ids) for ids in enc_prompt['input_ids']]

    labels = enc_full['input_ids'].clone()
    # Pad tokens -> -100
    labels[enc_full['attention_mask'] == 0] = -100
    # Right-padded: prompt is at the start; mask first prompt_len tokens.
    # But we set padding_side='left' for causal LM. With left padding the prompt is also at the
    # start of the *non-pad* segment. To keep the indexing simple we left-pad here too:
    # actually padding_side defaults to left; HF puts pads at the START.
    # So prompt occupies positions [pad_offset, pad_offset+prompt_len), answer is [pad_offset+prompt_len, end).
    # We want loss only on the answer part (the digit token + EOS). Mask everything before that.
    seq_len = labels.size(1)
    for i, plen in enumerate(prompt_lens):
        # number of pad tokens at the left:
        pad = (enc_full['attention_mask'][i] == 0).sum().item()
        labels[i, :pad + plen] = -100
    return {
        'input_ids': enc_full['input_ids'],
        'attention_mask': enc_full['attention_mask'],
        'labels': labels,
    }

def collate_predict(batch):
    """Tokenize prompt only; record the position to read logits at (last non-pad token)."""
    prompts = [render_prompt(b['title'], b['abstract'], with_label=False) for b in batch]
    enc = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_LEN,
                    return_tensors='pt', add_special_tokens=False)
    # With left-padding, the last non-pad position is always seq_len - 1
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'idx': torch.tensor([b['idx'] for b in batch], dtype=torch.long),
    }

print('tokenizer + prompt template ready, MAX_LEN =', MAX_LEN)

## 5. Model: 4-bit Qwen2.5-7B + LoRA

LoRA targets the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and FFN projections (`gate_proj`, `up_proj`, `down_proj`) — the standard "all linear layers" target list for Qwen-class models. Embeddings and output head stay frozen (and quantized).

The base model is loaded once at module level and **shared across folds**; each fold gets a fresh LoRA adapter via `get_peft_model`. This avoids reloading the 14GB base 5× per run.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print('loading base model in 4-bit (this takes ~30-60s on A100)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
)
base_model.config.use_cache = False  # required for grad checkpointing during training
base_model.gradient_checkpointing_enable()
base_model = prepare_model_for_kbit_training(base_model)
print('base model loaded; param count =', sum(p.numel() for p in base_model.parameters()))

LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

def fresh_lora_model():
    """Re-attach a fresh LoRA adapter to the (kept-in-memory) base model."""
    # Drop any previous adapter cleanly
    if hasattr(base_model, 'peft_config') and base_model.peft_config:
        try:
            base_model.unload()
        except Exception:
            pass
    model = get_peft_model(base_model, LORA_CONFIG)
    model.print_trainable_parameters()
    return model

print('LoRA factory ready (target_modules =', LORA_CONFIG.target_modules, ')')

## 6. Single-fold training + verbalizer inference

Standard SFT loss (cross-entropy, prompt masked) for training. For prediction we run a forward pass with the prompt only and read logits at the **last non-pad position** (which corresponds to the position where the model would emit the assistant's first token). We softmax over the 5 digit token ids and compute `expected = sum(p_i * i)`, returning a continuous score.

This makes the LLM output regression-style and stack-compatible with the encoder anchors.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 3            # SFT on 2k samples — 3 epochs is the standard sweet spot
LR = 2e-4             # LoRA standard
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
LABEL_TOKEN_TENSOR = torch.tensor(LABEL_TOKEN_IDS, dtype=torch.long, device=device)

@torch.no_grad()
def predict_verbalizer(model, loader):
    """Forward pass on prompt-only inputs; read logits at last non-pad position;
    softmax over the 5 digit tokens; return expected value sum(p_i * i)."""
    model.eval()
    n = len(loader.dataset)
    out = np.zeros(n, dtype=np.float32)
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(input_ids=ids, attention_mask=mask).logits  # (B, T, V)
        # With left-padding, last position is T-1 for every row
        last_logits = logits[:, -1, :]                                 # (B, V)
        digit_logits = last_logits[:, LABEL_TOKEN_TENSOR]              # (B, 5)
        probs = torch.softmax(digit_logits.float(), dim=-1)            # (B, 5)
        scores = (probs * torch.arange(1, 6, device=device).float()).sum(dim=-1)  # (B,)
        idx = batch['idx'].numpy()
        out[idx] = scores.cpu().numpy()
    if not np.isfinite(out).all():
        n_bad = int((~np.isfinite(out)).sum())
        print(f'  WARNING: predict produced {n_bad} non-finite scores; clamping to 3.0')
        out = np.where(np.isfinite(out), out, 3.0)
    return np.clip(out, 1.0, 5.0)


def train_one_fold(train_df, valid_df, public_df, private_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(PaperDataset(train_df, with_label=True),
                              batch_size=BATCH_TRAIN, shuffle=True,
                              collate_fn=collate_train, num_workers=2, pin_memory=False)
    valid_loader = DataLoader(PaperDataset(valid_df, with_label=False),
                              batch_size=BATCH_EVAL, shuffle=False,
                              collate_fn=collate_predict, num_workers=2, pin_memory=False)
    public_loader = DataLoader(PaperDataset(public_df.assign(Label=0), with_label=False),
                               batch_size=BATCH_EVAL, shuffle=False,
                               collate_fn=collate_predict, num_workers=2, pin_memory=False)
    private_loader = DataLoader(PaperDataset(private_df.assign(Label=0), with_label=False),
                                batch_size=BATCH_EVAL, shuffle=False,
                                collate_fn=collate_predict, num_workers=2, pin_memory=False)

    model = fresh_lora_model()
    # Only LoRA params have grad
    optim_params = [p for p in model.parameters() if p.requires_grad]
    optim = AdamW(optim_params, lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = (EPOCHS * len(train_loader)) // GRAD_ACCUM_STEPS
    scheduler = get_cosine_schedule_with_warmup(
        optim, num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps)

    best_state = None
    best_qwk = -1.0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        n_skipped = 0
        t0 = time.time()
        optim.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(device, non_blocking=True)
            mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
                loss = outputs.loss / GRAD_ACCUM_STEPS
            if not torch.isfinite(loss):
                n_skipped += 1
                optim.zero_grad(set_to_none=True)
                continue
            loss.backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(optim_params, GRAD_CLIP)
                optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)
            running += loss.item() * GRAD_ACCUM_STEPS
        val_scores = predict_verbalizer(model, valid_loader)
        rounded = np.clip(np.round(val_scores), 1, 5).astype(int)
        qwk = cohen_kappa_score(valid_df['Label'].astype(int).to_numpy(), rounded, weights='quadratic')
        skipped_msg = f'  skipped={n_skipped}' if n_skipped else ''
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={running/max(len(train_loader),1):.3f}  val_round_QWK={qwk:.4f}  ({time.time()-t0:.1f}s){skipped_msg}')
        if qwk > best_qwk:
            best_qwk = qwk
            # Save just the LoRA adapter weights to CPU (cheap)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                          if 'lora' in k.lower()}

    if best_state is None:
        raise RuntimeError('Training produced no usable epoch.')
    # Reload best LoRA weights for inference
    model.load_state_dict(best_state, strict=False)
    return (
        predict_verbalizer(model, valid_loader),
        predict_verbalizer(model, public_loader),
        predict_verbalizer(model, private_loader),
        best_qwk,
    )

print(f'train_one_fold ready (4-bit Qwen-7B + LoRA r=16, EPOCHS={EPOCHS}, LR={LR}, amp={"bf16" if USE_BF16 else "fp16"})')

## 7. Repeated CV (5 folds × 1 seed by default; bump to 3 seeds for full run)

Default is 1 seed (5 models, ~85-95 min on A100). Set `SEEDS = [252, 253, 254]` for the full 15-model run if time permits (~4.5h).

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLDS = 5
SEEDS = [252]              # default; extend to [252, 253, 254] for 15-model run
# SEEDS = [252, 253, 254]

y_class = train_full['Label'].astype(int).to_numpy()
oof_sum = np.zeros(len(train_full), dtype=np.float64)
oof_count = np.zeros(len(train_full), dtype=np.float64)
public_sum = np.zeros(len(public_full), dtype=np.float64)
private_sum = np.zeros(len(private_full), dtype=np.float64)
n_models = 0
fold_log = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_full, y_class), start=1):
        print(f'\n=== seed={seed} fold={fold}/{FOLDS} ===')
        t0 = time.time()
        train_df = train_full.iloc[tr_idx].reset_index(drop=True)
        valid_df = train_full.iloc[va_idx].reset_index(drop=True)
        val_scores, pub_scores, priv_scores, best_qwk = train_one_fold(
            train_df, valid_df, public_full, private_full, seed * 1000 + fold)
        oof_sum[va_idx] += val_scores
        oof_count[va_idx] += 1.0
        public_sum += pub_scores
        private_sum += priv_scores
        n_models += 1
        fold_log.append({'seed': seed, 'fold': fold, 'best_round_qwk': float(best_qwk),
                         'minutes': round((time.time()-t0)/60, 2)})

print('\n=== Done. trained', n_models, 'models. ===')
oof_scores = oof_sum / np.clip(oof_count, 1.0, None)
public_scores = public_sum / n_models
private_scores = private_sum / n_models
print('OOF round-QWK:',
      cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))

## 8. Constrained threshold tuning

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score, mean_absolute_error

TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()
DIST_PENALTY_LAMBDA = 0.5

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, oof, lambd=DIST_PENALTY_LAMBDA, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(oof, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds_constrained(y_class, oof_scores)
oof_pred = scores_to_labels(oof_scores, thresholds)
print('Constrained-tuned OOF QWK =', round(oof_qwk, 4))
print('  (anchors: E5_step12 0.6496, BGE 0.6404, SPECTER2 0.6373, SciNCL 0.6269, Ridge 0.5967)')
print('  (decision rule per L13: Qwen needs OOF >= 0.55 to clear signal floor)')
print('thresholds =', thresholds.tolist())
print('OOF predicted dist =', dict(zip([1,2,3,4,5], predicted_dist(oof_pred).round(3).tolist())))
print('TRAIN actual dist  =', dict(zip([1,2,3,4,5], TRAIN_DIST.round(3).tolist())))
print('OOF MAE =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))

## 9. Save artefacts

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'qwen_lora_finetune',
    'model': MODEL_NAME,
    'folds': FOLDS,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_len': MAX_LEN,
    'batch_train': BATCH_TRAIN,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'lr': LR,
    'lora_r': LORA_CONFIG.r,
    'lora_alpha': LORA_CONFIG.lora_alpha,
    'amp_dtype': 'bf16' if USE_BF16 else 'fp16',
    'inference': 'verbalizer (softmax over digit tokens, expected value)',
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'fold_log': fold_log,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': oof_scores, 'oof_pred': oof_pred}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores,
              'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores,
              'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'qwen_lora_finetune_submission.csv', index=False)
print('submission rows =', len(submission))

## 10. Zip + download (Colab / Kaggle / local)

In [ ]:
zip_path = pathlib.Path(OUT) / 'qwen_lora_finetune_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in pathlib.Path(RUN_DIR).iterdir():
        zf.write(p, arcname=f'qwen_lora_finetune/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))

if PLATFORM == 'colab':
    from google.colab import files
    files.download(str(zip_path))
elif PLATFORM == 'kaggle':
    import shutil
    kaggle_out = pathlib.Path('/kaggle/working') / zip_path.name
    if zip_path.resolve() != kaggle_out.resolve():
        shutil.copy(zip_path, kaggle_out)
    print('available at:', kaggle_out)
else:
    print('local run — zip is at:', zip_path)